# Code for GDL Project Submission

* please note that it is primarily meant to be run on google colab and might have to be adjusted if running locally

In [ ]:
!pip install torch_geometric
!pip install ogb

In [ ]:
import os
import urllib.request
import numpy as np
import scipy.sparse as sp
from scipy.io import loadmat

from ogb.nodeproppred import NodePropPredDataset
import gdown

YANDEx_BASE_URL = "https://raw.githubusercontent.com/yandex-research/heterophilous-graphs/main/data"

SNAP_PATENTS_FILE_ID = "1ldh23TSY1PwXia6dU0MYcpyEgX-w3Hia"


def _normalize_adj(adj: sp.csr_matrix) -> sp.csr_matrix:
    deg = np.array(adj.sum(1)).flatten()
    deg_inv_sqrt = np.zeros_like(deg, dtype=np.float32)
    nonzero = deg > 0
    deg_inv_sqrt[nonzero] = np.power(deg[nonzero], -0.5)
    D = sp.diags(deg_inv_sqrt)
    return (D @ adj @ D).tocsr()


def _make_directed_adjs(row, col, n, add_self_loops=True, normalize_adj=True):
    adj_out = sp.coo_matrix(
        (np.ones(len(row), dtype=np.float32), (row, col)),
        shape=(n, n),
    ).tocsr()
    adj_out.data[:] = 1.0

    adj_in = adj_out.transpose().tocsr()
    adj_in.data[:] = 1.0

    adj_all = (adj_out + adj_in).tocsr()
    adj_all.data[:] = 1.0

    if add_self_loops:
        I = sp.eye(n, dtype=np.float32, format="csr")
        adj_out = adj_out + I
        adj_in = adj_in + I
        adj_all = adj_all + I

    adj_out.data[:] = 1.0
    adj_in.data[:] = 1.0
    adj_all.data[:] = 1.0

    if normalize_adj:
        adj_out = _normalize_adj(adj_out)
        adj_in = _normalize_adj(adj_in)
        adj_all = _normalize_adj(adj_all)

    return adj_out, adj_in, adj_all


def _quantile_labels(values, nclasses=5):
    values = np.asarray(values).reshape(-1)
    qs = np.quantile(values, np.linspace(0, 1, nclasses + 1))
    labels = np.digitize(values, qs[1:-1], right=True)
    return labels.astype(np.int64)


def _load_yandex_directed(
    name,
    root="data/heterophilous-graphs",
    split_index=0,
    normalize_adj=True,
    add_self_loops=False,
):
    if "directed" not in name:
        raise ValueError(f"{name} is not a directed Yandex dataset name.")

    os.makedirs(root, exist_ok=True)
    filepath = os.path.join(root, f"{name}.npz")

    if not os.path.exists(filepath):
        url = f"{YANDEx_BASE_URL}/{name}.npz"
        print(f"Downloading {url} -> {filepath}")
        urllib.request.urlretrieve(url, filepath)

    data = np.load(filepath)

    x = data["node_features"].astype(np.float32)
    features = sp.csr_matrix(x)
    labels = data["node_labels"].astype(np.int64)

    edges = data["edges"]
    row = edges[:, 0]
    col = edges[:, 1]
    n = x.shape[0]

    adj_out, adj_in, adj_all = _make_directed_adjs(
        row=row,
        col=col,
        n=n,
        add_self_loops=add_self_loops,
        normalize_adj=normalize_adj,
    )

    train_masks = data["train_masks"]
    val_masks = data["val_masks"]
    test_masks = data["test_masks"]

    if train_masks.ndim == 2:
        train_idx = np.where(train_masks[split_index].astype(bool))[0]
        val_idx = np.where(val_masks[split_index].astype(bool))[0]
        test_idx = np.where(test_masks[split_index].astype(bool))[0]
    else:
        train_idx = np.where(train_masks.astype(bool))[0]
        val_idx = np.where(val_masks.astype(bool))[0]
        test_idx = np.where(test_masks.astype(bool))[0]

    return features, labels, adj_out, adj_in, adj_all, train_idx, val_idx, test_idx


def _load_snap_patents(
    root="data/snap_patents",
    split_seed=42,
    normalize_adj=True,
    add_self_loops=True,
):
    os.makedirs(root, exist_ok=True)
    mat_path = os.path.join(root, "snap_patents.mat")

    if not os.path.exists(mat_path):
        print("Downloading SNAP patents...")
        gdown.download(
            f"https://drive.google.com/uc?id={SNAP_PATENTS_FILE_ID}",
            mat_path,
            quiet=False,
        )

    data = loadmat(mat_path)

    edge_index = data["edge_index"]
    node_feat = data["node_feat"]
    years = data["years"].flatten()

    features = node_feat.tocsr().astype(np.float32)
    labels = _quantile_labels(years, nclasses=5)

    if edge_index.shape[0] != 2:
        raise ValueError(f"Unexpected edge_index shape for snap-patents: {edge_index.shape}")

    row, col = edge_index
    n = features.shape[0]

    adj_out, adj_in, adj_all = _make_directed_adjs(
        row=row,
        col=col,
        n=n,
        add_self_loops=add_self_loops,
        normalize_adj=normalize_adj,
    )

    rng = np.random.default_rng(split_seed)
    idx = rng.permutation(n)
    train_idx = idx[: int(0.6 * n)]
    val_idx = idx[int(0.6 * n): int(0.8 * n)]
    test_idx = idx[int(0.8 * n):]

    return features, labels, adj_out, adj_in, adj_all, train_idx, val_idx, test_idx


def _load_arxiv_year(
    root="data/ogbn_arxiv",
    split_seed=42,
    normalize_adj=True,
    add_self_loops=True,
):
    import torch
    from ogb.nodeproppred import NodePropPredDataset

    original_torch_load = torch.load

    def torch_load_compat(*args, **kwargs):
        if "weights_only" not in kwargs:
            kwargs["weights_only"] = False
        return original_torch_load(*args, **kwargs)

    torch.load = torch_load_compat
    try:
        dataset = NodePropPredDataset(name="ogbn-arxiv", root=root)
        graph, _ = dataset[0]
    finally:
        torch.load = original_torch_load

    features = sp.csr_matrix(graph["node_feat"].astype(np.float32))
    years = graph["node_year"].flatten()
    labels = _quantile_labels(years, nclasses=5)

    edge_index = graph["edge_index"]
    row = edge_index[0]
    col = edge_index[1]
    n = graph["num_nodes"]

    adj_out, adj_in, adj_all = _make_directed_adjs(
        row=row,
        col=col,
        n=n,
        add_self_loops=add_self_loops,
        normalize_adj=normalize_adj,
    )

    rng = np.random.default_rng(split_seed)
    idx = rng.permutation(n)
    train_idx = idx[: int(0.6 * n)]
    val_idx = idx[int(0.6 * n): int(0.8 * n)]
    test_idx = idx[int(0.8 * n):]

    return features, labels, adj_out, adj_in, adj_all, train_idx, val_idx, test_idx


def load_directed_dataset(
    name,
    root="data",
    split_index=0,
    split_seed=42,
    normalize_adj=True,
    add_self_loops=True,
):
    if name in {"chameleon_filtered_directed", "squirrel_filtered_directed"}:
        return _load_yandex_directed(
            name=name,
            root=os.path.join(root, "heterophilous-graphs"),
            split_index=split_index,
            normalize_adj=normalize_adj,
            add_self_loops=add_self_loops,
        )

    if name == "snap-patents":
        return _load_snap_patents(
            root=os.path.join(root, "snap_patents"),
            split_seed=split_seed,
            normalize_adj=normalize_adj,
            add_self_loops=add_self_loops,
        )

    if name == "arxiv-year":
        return _load_arxiv_year(
            root=os.path.join(root, "ogbn_arxiv"),
            split_seed=split_seed,
            normalize_adj=normalize_adj,
            add_self_loops=add_self_loops,
        )

    raise ValueError(
        f"Unsupported dataset: {name}. "
        f"Choose from: chameleon_filtered_directed, squirrel_filtered_directed, snap-patents, arxiv-year"
    )

In [ ]:
import numpy as np
import pandas as pd


def compute_dataset_statistics(name, root="data", split_index=0, split_seed=42):
    features, labels, adj_out, adj_in, adj_all, train_idx, val_idx, test_idx = load_directed_dataset(
        name=name,
        root=root,
        split_index=split_index,
        split_seed=split_seed,
        normalize_adj=False,
        add_self_loops=False,
    )

    labels = np.asarray(labels).reshape(-1)

    num_nodes = features.shape[0]

    num_edges = adj_out.nnz

    num_classes = len(np.unique(labels))

    row, col = adj_out.nonzero()
    if len(row) > 0:
        homophily = np.mean(labels[row] == labels[col])
    else:
        homophily = float("nan")

    reverse_exists = adj_out[col, row].A1 > 0
    asymmetry = (1.0 - np.mean(reverse_exists) if len(row) > 0 else float("nan"))*100

    return {
        "dataset": name,
        "num_nodes": int(num_nodes),
        "num_edges": int(num_edges),
        "num_classes": int(num_classes),
        "homophily": float(homophily),
        "asymmetry": float(asymmetry),
    }


def compute_all_dataset_statistics(
    dataset_names=None,
    root="data",
    split_index=0,
    split_seed=42,
    round_digits=3,
):
    if dataset_names is None:
        dataset_names = [
            "snap-patents",
            "arxiv-year",
            "chameleon_filtered_directed",
            "squirrel_filtered_directed",
        ]

    rows = []
    for name in dataset_names:
        stats = compute_dataset_statistics(
            name=name,
            root=root,
            split_index=split_index,
            split_seed=split_seed,
        )
        rows.append(stats)

    df = pd.DataFrame(rows)

    df["homophily"] = df["homophily"].round(round_digits)
    df["asymmetry"] = df["asymmetry"].round(round_digits)

    return df

df_stats = compute_all_dataset_statistics()
print(df_stats.to_latex(index=False, escape=False))

In [ ]:
def scipy_to_torch_sparse(mat, device, binarize=False):
    mat = mat.tocoo()
    indices = torch.tensor([mat.row, mat.col], dtype=torch.long).to(device)
    values = torch.ones_like(torch.tensor(mat.data)) if binarize else torch.tensor(mat.data)
    values = values.float().to(device)
    return torch.sparse_coo_tensor(indices, values, mat.shape).coalesce()

def sparse_dropout(x, p, training):
    if not training or p <= 0:
        return x
    x = x.coalesce()
    mask = torch.rand(x.values().shape, device=x.device) > p
    return torch.sparse_coo_tensor(
        x.indices()[:, mask],
        x.values()[mask] / (1 - p),
        x.shape
    ).coalesce()

In [ ]:
import torch
import numpy as np


def load_directed_dataset_torch(
    name,
    root="data",
    split_index=0,
    split_seed=42,
    normalize_adj=True,
    add_self_loops=True,
):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    features, labels, adj_out, adj_in, adj_all, train_idx, val_idx, test_idx = load_directed_dataset(
        name=name,
        root=root,
        split_index=split_index,
        split_seed=split_seed,
        normalize_adj=normalize_adj,
        add_self_loops=add_self_loops,
    )

    # Convert features + labels
    x = scipy_to_torch_sparse(features, device)
    y = torch.tensor(labels, dtype=torch.long, device=device)

    # Convert adjacencies
    adj_out_t = scipy_to_torch_sparse(adj_out, device)
    adj_in_t = scipy_to_torch_sparse(adj_in, device)
    adj_all_t = scipy_to_torch_sparse(adj_all, device)

    # Indices
    train_idx = torch.tensor(train_idx, dtype=torch.long, device=device)
    val_idx = torch.tensor(val_idx, dtype=torch.long, device=device)
    test_idx = torch.tensor(test_idx, dtype=torch.long, device=device)

    return x, y, adj_out_t, adj_in_t, adj_all_t, train_idx, val_idx, test_idx

## Models

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
# parser and layer for mixhop

class AdjacencyPowersParser:
    def __init__(self, spec: str):
        parts = spec.split(",")
        self._powers = []
        self._ratios = []
        has_colon = None

        for i, part in enumerate(parts):
            current_has_colon = ":" in part
            if i == 0:
                has_colon = current_has_colon
            elif has_colon != current_has_colon:
                raise ValueError("Either all adjacency powers must use ':' or none of them should.")

            fields = part.split(":")
            self._powers.append(int(fields[0]))
            self._ratios.append([float(x) for x in fields[1:]] if has_colon else [1.0])

    @property
    def powers(self):
        return list(self._powers)

    def output_capacity(self, num_classes: int) -> int:
        if all(len(r) == 1 and r[0] == 1 for r in self._ratios):
            return num_classes * len(self._powers)
        return int(sum(r[-1] for r in self._ratios))

    def divide_capacity(self, layer_index: int, total_dim: int):
        sizes = [r[min(layer_index, len(r) - 1)] for r in self._ratios]
        unit = total_dim / float(np.sum(sizes))
        dims = [int(np.round(s * unit)) for s in sizes[:-1]]
        dims.append(int(total_dim - sum(dims)))
        return dims


class PSumHead(nn.Module):
    """Weighted segment-sum output layer from the original code."""

    def __init__(self, in_dim: int, num_classes: int):
        super().__init__()
        self.num_classes = num_classes
        self.num_segments = in_dim // num_classes
        self.wasted_capacity = in_dim % num_classes
        if self.num_segments == 0:
            raise ValueError("Input dimension must be >= num_classes for PSumHead")
        self.segment_logits = nn.Parameter(torch.zeros(self.num_segments))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        usable = self.num_segments * self.num_classes
        x = x[:, :usable].reshape(x.shape[0], self.num_segments, self.num_classes)
        weights = torch.softmax(self.segment_logits, dim=0)
        return (x * weights.view(1, -1, 1)).sum(dim=1)

    def regularization_loss(self) -> torch.Tensor:
        return 1e-3 * self.segment_logits.pow(2).mean()

In [ ]:
# MLP

import torch
import torch.nn as nn
import torch.nn.functional as F


class MLPNet(nn.Module):
    def __init__(
        self,
        input_dim: int,
        num_classes: int,
        hidden_dim: int = 64,
        num_layers: int = 2,
        input_dropout: float = 0.5,
        dropout: float = 0.5,
        nonlinearity: str = "relu",
    ):
        super().__init__()
        if num_layers < 1:
            raise ValueError("num_layers must be >= 1")

        self.input_dropout = input_dropout
        self.dropout = dropout
        self.nonlinearity = nonlinearity.lower()

        self.layers = nn.ModuleList()
        self.norms = nn.ModuleList()

        if num_layers == 1:
            self.layers.append(nn.Linear(input_dim, num_classes, bias=False))
        else:
            self.layers.append(nn.Linear(input_dim, hidden_dim, bias=False))
            for _ in range(num_layers - 2):
                self.layers.append(nn.Linear(hidden_dim, hidden_dim, bias=False))
                self.norms.append(nn.BatchNorm1d(hidden_dim))
            self.layers.append(nn.Linear(hidden_dim, num_classes, bias=False))

            if num_layers == 2:
                self.norms.append(nn.BatchNorm1d(hidden_dim))

    def activation(self, x: torch.Tensor) -> torch.Tensor:
        if self.nonlinearity == "relu":
            return F.relu(x)
        if self.nonlinearity == "tanh":
            return torch.tanh(x)
        if self.nonlinearity == "gelu":
            return F.gelu(x)
        raise ValueError(f"Unsupported nonlinearity: {self.nonlinearity}")

    def forward(self, x_sparse: torch.Tensor) -> torch.Tensor:
        x_sparse = sparse_dropout(x_sparse, self.input_dropout, self.training)
        x = x_sparse.to_dense()
        x = F.normalize(x, p=2, dim=1)

        for i, layer in enumerate(self.layers):
            if i > 0:
                x = F.dropout(x, p=self.dropout, training=self.training)

            x = layer(x)

            if i != len(self.layers) - 1:
                x = self.activation(self.norms[i](x))

        return x

In [ ]:
# GCN

class GCNLayer(nn.Module):
    def __init__(self, in_dim: int, out_dim: int, bias: bool = False):
        super().__init__()
        self.linear = nn.Linear(in_dim, out_dim, bias=bias)

    def forward(self, x: torch.Tensor, adj: torch.Tensor) -> torch.Tensor:
        x = self.linear(x)
        x = torch.sparse.mm(adj, x)
        return x


class GCNNet(nn.Module):
    def __init__(
        self,
        input_dim: int,
        num_classes: int,
        hidden_dim: int = 64,
        num_layers: int = 2,
        input_dropout: float = 0.5,
        dropout: float = 0.5,
        nonlinearity: str = "relu",
    ):
        super().__init__()
        if num_layers < 1:
            raise ValueError("num_layers must be >= 1")

        self.input_dropout = input_dropout
        self.dropout = dropout
        self.nonlinearity = nonlinearity.lower()

        self.layers = nn.ModuleList()
        self.norms = nn.ModuleList()

        if num_layers == 1:
            self.layers.append(GCNLayer(input_dim, num_classes))
        else:
            self.layers.append(GCNLayer(input_dim, hidden_dim))
            for _ in range(num_layers - 2):
                self.layers.append(GCNLayer(hidden_dim, hidden_dim))
                self.norms.append(nn.BatchNorm1d(hidden_dim))
            self.layers.append(GCNLayer(hidden_dim, num_classes))

            if num_layers == 2:
                self.norms.append(nn.BatchNorm1d(hidden_dim))

    def activation(self, x: torch.Tensor) -> torch.Tensor:
        if self.nonlinearity == "relu":
            return F.relu(x)
        if self.nonlinearity == "tanh":
            return torch.tanh(x)
        if self.nonlinearity == "gelu":
            return F.gelu(x)
        raise ValueError(f"Unsupported nonlinearity: {self.nonlinearity}")

    def forward(self, x_sparse: torch.Tensor, adj: torch.Tensor) -> torch.Tensor:
        x_sparse = sparse_dropout(x_sparse, self.input_dropout, self.training)
        x = x_sparse.to_dense()
        x = F.normalize(x, p=2, dim=1)

        for i, layer in enumerate(self.layers):
            if i > 0:
                x = F.dropout(x, p=self.dropout, training=self.training)

            x = layer(x, adj)

            if i != len(self.layers) - 1:
                x = self.activation(self.norms[i](x))

        return x

In [ ]:
# baseline mixhop

class MixHopConv(nn.Module):
    def __init__(self, in_dim: int, adjacency_powers, dim_per_power):
        super().__init__()
        if len(adjacency_powers) != len(dim_per_power):
            raise ValueError("adjacency_powers and dim_per_power must have the same length")
        self.adjacency_powers = list(adjacency_powers)
        self.projections = nn.ModuleList([nn.Linear(in_dim, d, bias=False) for d in dim_per_power])

    def forward(self, x: torch.Tensor, adj: torch.Tensor) -> torch.Tensor:
        cached = {0: x}
        max_power = max(self.adjacency_powers)
        for p in range(1, max_power + 1):
            cached[p] = torch.sparse.mm(adj, cached[p - 1])
        parts = [proj(cached[p]) for p, proj in zip(self.adjacency_powers, self.projections)]
        return torch.cat(parts, dim=1)


class MixHopNet(nn.Module):
    def __init__(
        self,
        input_dim: int,
        num_classes: int,
        hidden_dims_csv: str,
        adj_pows: str,
        nonlinearity: str = "relu",
        output_layer: str = "wsum",
        input_dropout: float = 0.2,
        layer_dropout: float = 0.2,
    ):
        super().__init__()
        self.input_dropout = input_dropout
        self.layer_dropout = layer_dropout
        self.nonlinearity = nonlinearity.lower()
        self.output_layer = output_layer.lower()

        parser = AdjacencyPowersParser(adj_pows)
        hidden_dims = [int(x) for x in hidden_dims_csv.split(",") if x]
        layer_dims = hidden_dims + [parser.output_capacity(num_classes)]

        self.layers = nn.ModuleList()
        self.norms = nn.ModuleList()
        prev_dim = input_dim
        for i, total_dim in enumerate(layer_dims):
            per_power = parser.divide_capacity(i, total_dim)
            self.layers.append(MixHopConv(prev_dim, parser.powers, per_power))
            prev_dim = sum(per_power)
            if i != len(layer_dims) - 1:
                self.norms.append(nn.BatchNorm1d(prev_dim))

        if self.output_layer == "wsum":
            self.head = PSumHead(prev_dim, num_classes)
        elif self.output_layer == "fc":
            self.head = nn.Linear(prev_dim, num_classes, bias=False)
        else:
            raise ValueError("output_layer must be 'wsum' or 'fc'")

    def activation(self, x: torch.Tensor) -> torch.Tensor:
        if self.nonlinearity == "relu":
            return F.relu(x)
        if self.nonlinearity == "tanh":
            return torch.tanh(x)
        if self.nonlinearity == "gelu":
            return F.gelu(x)
        raise ValueError(f"Unsupported nonlinearity: {self.nonlinearity}")

    def forward(self, x_sparse: torch.Tensor, adj: torch.Tensor) -> torch.Tensor:
        x_sparse = sparse_dropout(x_sparse, self.input_dropout, self.training)
        x = x_sparse.to_dense()
        x = F.normalize(x, p=2, dim=1)

        for i, layer in enumerate(self.layers):
            if i > 0:
                x = F.dropout(x, p=self.layer_dropout, training=self.training)
            x = layer(x, adj)
            if i != len(self.layers) - 1:
                x = self.activation(self.norms[i](x))

        return self.head(x)

In [ ]:
# dir only

class DirMixHopConv(nn.Module):
    def __init__(self, in_dim: int, adjacency_powers, dim_per_power, alpha=0.5):
        super().__init__()
        if len(adjacency_powers) != len(dim_per_power):
            raise ValueError("adjacency_powers and dim_per_power must have the same length")

        self.adjacency_powers = list(adjacency_powers)
        self.alpha = alpha

        self.self_projections = nn.ModuleDict()
        self.out_projections = nn.ModuleDict()
        self.in_projections = nn.ModuleDict()

        for p, d in zip(self.adjacency_powers, dim_per_power):
            key = str(p)
            if p == 0:
                self.self_projections[key] = nn.Linear(in_dim, d, bias=False)
            else:
                self.out_projections[key] = nn.Linear(in_dim, d, bias=False)
                self.in_projections[key] = nn.Linear(in_dim, d, bias=False)

    def _compute_powers(self, x: torch.Tensor, adj: torch.Tensor, max_power: int):
        cached = {0: x}
        for p in range(1, max_power + 1):
            cached[p] = torch.sparse.mm(adj, cached[p - 1])
        return cached

    def forward(self, x: torch.Tensor, adj_out: torch.Tensor, adj_in: torch.Tensor) -> torch.Tensor:
        max_power = max(self.adjacency_powers)

        cached_out = self._compute_powers(x, adj_out, max_power)
        cached_in = self._compute_powers(x, adj_in, max_power)

        alpha = self.alpha
        if isinstance(alpha, torch.Tensor):
            alpha = torch.clamp(alpha, 0.0, 1.0)

        parts = []
        for p in self.adjacency_powers:
            key = str(p)
            if p == 0:
                part = self.self_projections[key](cached_out[0])  # just x
            else:
                out_part = self.out_projections[key](cached_out[p])
                in_part = self.in_projections[key](cached_in[p])
                part = alpha * out_part + (1.0 - alpha) * in_part
            parts.append(part)

        return torch.cat(parts, dim=1)


class DirMixHopNet(nn.Module):
    def __init__(
        self,
        input_dim: int,
        num_classes: int,
        hidden_dims_csv: str,
        adj_pows: str,
        nonlinearity: str = "relu",
        output_layer: str = "wsum",
        input_dropout: float = 0.2,
        layer_dropout: float = 0.2,
        alpha: float = 0.5,
        learn_alpha: bool = True,
    ):
        super().__init__()
        self.input_dropout = input_dropout
        self.layer_dropout = layer_dropout
        self.nonlinearity = nonlinearity.lower()
        self.output_layer = output_layer.lower()

        if learn_alpha:
            self.alpha = nn.Parameter(torch.tensor(float(alpha)))
        else:
            self.register_buffer("alpha", torch.tensor(float(alpha)), persistent=False)

        parser = AdjacencyPowersParser(adj_pows)
        hidden_dims = [int(x) for x in hidden_dims_csv.split(",") if x]
        layer_dims = hidden_dims + [parser.output_capacity(num_classes)]

        self.layers = nn.ModuleList()
        self.norms = nn.ModuleList()
        prev_dim = input_dim

        for i, total_dim in enumerate(layer_dims):
            per_power = parser.divide_capacity(i, total_dim)
            self.layers.append(
                DirMixHopConv(
                    in_dim=prev_dim,
                    adjacency_powers=parser.powers,
                    dim_per_power=per_power,
                    alpha=self.alpha,
                )
            )
            prev_dim = sum(per_power)
            if i != len(layer_dims) - 1:
                self.norms.append(nn.BatchNorm1d(prev_dim))

        if self.output_layer == "wsum":
            self.head = PSumHead(prev_dim, num_classes)
        elif self.output_layer == "fc":
            self.head = nn.Linear(prev_dim, num_classes, bias=False)
        else:
            raise ValueError("output_layer must be 'wsum' or 'fc'")

    def activation(self, x: torch.Tensor) -> torch.Tensor:
        if self.nonlinearity == "relu":
            return F.relu(x)
        if self.nonlinearity == "tanh":
            return torch.tanh(x)
        if self.nonlinearity == "gelu":
            return F.gelu(x)
        raise ValueError(f"Unsupported nonlinearity: {self.nonlinearity}")

    def forward(
        self,
        x_sparse: torch.Tensor,
        adj_out: torch.Tensor,
        adj_in: torch.Tensor = None,
    ) -> torch.Tensor:
        x_sparse = sparse_dropout(x_sparse, self.input_dropout, self.training)
        x = x_sparse.to_dense()
        x = F.normalize(x, p=2, dim=1)

        if adj_in is None:
            adj_in = adj_out.transpose(0, 1).coalesce()

        for i, layer in enumerate(self.layers):
            if i > 0:
                x = F.dropout(x, p=self.layer_dropout, training=self.training)
            x = layer(x, adj_out, adj_in)
            if i != len(self.layers) - 1:
                x = self.activation(self.norms[i](x))

        return self.head(x)

In [ ]:
# dir and undir

class DirMixHopConv(nn.Module):
    def __init__(self, in_dim: int, adjacency_powers, dim_per_power, alpha=0.5):
        super().__init__()
        if len(adjacency_powers) != len(dim_per_power):
            raise ValueError("adjacency_powers and dim_per_power must have the same length")

        self.adjacency_powers = list(adjacency_powers)
        self.alpha = alpha

        self.self_projections = nn.ModuleDict()
        self.out_projections = nn.ModuleDict()
        self.in_projections = nn.ModuleDict()
        self.all_projections = nn.ModuleDict()  # NEW

        for p, d in zip(self.adjacency_powers, dim_per_power):
            key = str(p)
            if p == 0:
                self.self_projections[key] = nn.Linear(in_dim, d, bias=False)
            else:
                self.out_projections[key] = nn.Linear(in_dim, d, bias=False)
                self.in_projections[key] = nn.Linear(in_dim, d, bias=False)
                self.all_projections[key] = nn.Linear(in_dim, d, bias=False)

    def _compute_powers(self, x: torch.Tensor, adj: torch.Tensor, max_power: int):
        cached = {0: x}
        for p in range(1, max_power + 1):
            cached[p] = torch.sparse.mm(adj, cached[p - 1])
        return cached

    def forward(self, x, adj_out, adj_in, adj_all):
        max_power = max(self.adjacency_powers)

        cached_out = self._compute_powers(x, adj_out, max_power)
        cached_in = self._compute_powers(x, adj_in, max_power)
        cached_all = self._compute_powers(x, adj_all, max_power)

        alpha = self.alpha
        if isinstance(alpha, torch.Tensor):
            alpha = torch.clamp(alpha, 0.0, 1.0)

        # weights: directional vs undirected
        w_out = alpha * 0.5
        w_in = alpha * 0.5
        w_all = (1.0 - alpha)

        parts = []
        for p in self.adjacency_powers:
            key = str(p)

            if p == 0:
                part = self.self_projections[key](x)
            else:
                out_part = self.out_projections[key](cached_out[p])
                in_part = self.in_projections[key](cached_in[p])
                all_part = self.all_projections[key](cached_all[p])

                part = w_out * out_part + w_in * in_part + w_all * all_part

            parts.append(part)

        return torch.cat(parts, dim=1)


class DirMixHopNet(nn.Module):
    def __init__(
        self,
        input_dim: int,
        num_classes: int,
        hidden_dims_csv: str,
        adj_pows: str,
        nonlinearity: str = "relu",
        output_layer: str = "wsum",
        input_dropout: float = 0.2,
        layer_dropout: float = 0.2,
        alpha: float = 0.5,
        learn_alpha: bool = True,
    ):
        super().__init__()
        self.input_dropout = input_dropout
        self.layer_dropout = layer_dropout
        self.nonlinearity = nonlinearity.lower()
        self.output_layer = output_layer.lower()

        if learn_alpha:
            self.alpha = nn.Parameter(torch.tensor(float(alpha)))
        else:
            self.register_buffer("alpha", torch.tensor(float(alpha)), persistent=False)

        parser = AdjacencyPowersParser(adj_pows)
        hidden_dims = [int(x) for x in hidden_dims_csv.split(",") if x]
        layer_dims = hidden_dims + [parser.output_capacity(num_classes)]

        self.layers = nn.ModuleList()
        self.norms = nn.ModuleList()
        prev_dim = input_dim

        for i, total_dim in enumerate(layer_dims):
            per_power = parser.divide_capacity(i, total_dim)
            self.layers.append(
                DirMixHopConv(
                    in_dim=prev_dim,
                    adjacency_powers=parser.powers,
                    dim_per_power=per_power,
                    alpha=self.alpha,
                )
            )
            prev_dim = sum(per_power)
            if i != len(layer_dims) - 1:
                self.norms.append(nn.BatchNorm1d(prev_dim))

        if self.output_layer == "wsum":
            self.head = PSumHead(prev_dim, num_classes)
        elif self.output_layer == "fc":
            self.head = nn.Linear(prev_dim, num_classes, bias=False)
        else:
            raise ValueError("output_layer must be 'wsum' or 'fc'")

    def activation(self, x: torch.Tensor) -> torch.Tensor:
        if self.nonlinearity == "relu":
            return F.relu(x)
        if self.nonlinearity == "tanh":
            return torch.tanh(x)
        if self.nonlinearity == "gelu":
            return F.gelu(x)
        raise ValueError(f"Unsupported nonlinearity: {self.nonlinearity}")

    def forward(
        self,
        x_sparse: torch.Tensor,
        adj_out: torch.Tensor,
        adj_in: torch.Tensor = None,
        adj_all: torch.Tensor = None,
    ):
        x_sparse = sparse_dropout(x_sparse, self.input_dropout, self.training)
        x = x_sparse.to_dense()
        x = F.normalize(x, p=2, dim=1)

        if adj_in is None:
            adj_in = adj_out.transpose(0, 1).coalesce()

        if adj_all is None:
            adj_all = (adj_out + adj_in).coalesce()

        for i, layer in enumerate(self.layers):
            if i > 0:
                x = F.dropout(x, p=self.layer_dropout, training=self.training)

            x = layer(x, adj_out, adj_in, adj_all)

            if i != len(self.layers) - 1:
                x = self.activation(self.norms[i](x))

        return self.head(x)

## Training

In [ ]:
import os
import json
import random
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from pathlib import Path

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

def make_reproducible(seed=42):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def accuracy_from_logits(logits, labels, idx):
    pred = logits[idx].argmax(1)
    return (pred == labels[idx]).float().mean().item()


def build_model(model_name, config, input_dim, num_classes, device):
    if model_name == "mlp":
        model = MLPNet(
            input_dim=input_dim,
            num_classes=num_classes,
            hidden_dim=config["hidden_dim"],
            num_layers=config.get("num_layers", 2),
            input_dropout=config.get("input_dropout", 0.5),
            dropout=config.get("dropout", 0.5),
            nonlinearity=config.get("nonlinearity", "relu"),
        )
    elif model_name == "gcn":
        model = GCNNet(
            input_dim=input_dim,
            num_classes=num_classes,
            hidden_dim=config["hidden_dim"],
            num_layers=config.get("num_layers", 2),
            input_dropout=config.get("input_dropout", 0.5),
            dropout=config.get("dropout", 0.5),
            nonlinearity=config.get("nonlinearity", "relu"),
        )
    elif model_name == "mixhop":
        model = MixHopNet(
            input_dim=input_dim,
            num_classes=num_classes,
            hidden_dims_csv=config["hidden_dims_csv"],
            adj_pows=config.get("adj_pows", "0,1,2"),
            output_layer=config.get("output_layer", "fc"),
            input_dropout=config.get("input_dropout", 0.5),
            layer_dropout=config.get("layer_dropout", 0.5),
            nonlinearity=config.get("nonlinearity", "relu"),
        )
    elif model_name == "dir_mixhop":
        model = DirMixHopNet(
            input_dim=input_dim,
            num_classes=num_classes,
            hidden_dims_csv=config["hidden_dims_csv"],
            adj_pows=config.get("adj_pows", "0,1,2"),
            output_layer=config.get("output_layer", "fc"),
            input_dropout=config.get("input_dropout", 0.5),
            layer_dropout=config.get("layer_dropout", 0.5),
            nonlinearity=config.get("nonlinearity", "relu"),
            alpha=config.get("alpha", 0.5),
            learn_alpha=config.get("learn_alpha", True),
        )
    else:
        raise ValueError(model_name)

    return model.to(device)


def run_one(
    dataset_name,
    model_name,
    config,
    split_index=0,
    split_seed=42,
    seed=0,
    max_epochs=400,
    patience=80,
    root="data",
    verbose=False,
):
    make_reproducible(seed)

    x, y, adj_out_t, adj_in_t, adj_all_t, train_idx, val_idx, test_idx = load_directed_dataset_torch(
        name=dataset_name,
        root=root,
        split_index=split_index,
        split_seed=split_seed,
        normalize_adj=True,
        add_self_loops=True,
    )

    model = build_model(
        model_name=model_name,
        config=config,
        input_dim=x.shape[1],
        num_classes=int(y.max().item()) + 1,
        device=y.device,
    )

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=config["lr"],
        weight_decay=config.get("weight_decay", 0.0),
    )

    total_params = sum(p.numel() for p in model.parameters())

    best_val = -1.0
    best_test = -1.0
    best_train = -1.0
    best_epoch = -1
    bad_epochs = 0

    for epoch in range(max_epochs):
        model.train()
        optimizer.zero_grad()

        if model_name == "mlp":
            logits = model(x)
        elif model_name == "gcn":
            logits = model(x, adj_all_t)
        elif model_name == "mixhop":
            logits = model(x, adj_all_t)
        elif model_name == "dir_mixhop":
            logits = model(x, adj_out_t, adj_in_t, adj_all_t) # or just in and out if dir only
        else:
            raise ValueError(model_name)

        loss = F.cross_entropy(logits[train_idx], y[train_idx])
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            if model_name == "mlp":
                logits = model(x)
            elif model_name == "gcn":
                logits = model(x, adj_all_t)
            elif model_name == "mixhop":
                logits = model(x, adj_all_t)
            elif model_name == "dir_mixhop":
                logits = model(x, adj_out_t, adj_in_t, adj_all_t) # or just in and out if dir only

            train_acc = accuracy_from_logits(logits, y, train_idx)
            val_acc = accuracy_from_logits(logits, y, val_idx)
            test_acc = accuracy_from_logits(logits, y, test_idx)

        if val_acc > best_val:
            best_val = val_acc
            best_test = test_acc
            best_train = train_acc
            best_epoch = epoch
            bad_epochs = 0
        else:
            bad_epochs += 1

        if verbose and epoch % 20 == 0:
            print(f"{model_name} | epoch={epoch} loss={loss:.4f} train={train_acc:.3f} val={val_acc:.3f} test={test_acc:.3f}")

        if bad_epochs >= patience:
            break

    return {
        "dataset": dataset_name,
        "model": model_name,
        "split_index": split_index,
        "split_seed": split_seed,
        "seed": seed,
        "best_train": best_train,
        "best_val": best_val,
        "best_test": best_test,
        "best_epoch": best_epoch,
        "epochs_ran": epoch + 1,
        "params": total_params,
        "config": json.dumps(config, sort_keys=True),
        "alpha": float(model.alpha.item()),
    }

In [ ]:
CURRENT_DATASET = "arxiv-year"
CURRENT_ROOT = "data"

# For large datasets with random splits
CURRENT_SPLIT_SEED = 0

# For Yandex datasets with official splits
TUNE_SPLIT_INDEX = 0

# Seeds for quick manual tuning
TUNE_SEEDS = [0]

# Seeds for final experiment
FINAL_SEEDS = [0, 1]

# Whether to use all official splits in final runs
USE_ALL_SPLITS_FOR_FINAL = True


HYPERPARAMS = {
    "mlp": {
        "hidden_dim": 64,
        "num_layers": 2,
        "input_dropout": 0.5,
        "dropout": 0.5,
        "nonlinearity": "relu",
        "lr": 5e-4,
        "weight_decay": 1e-4,
    },
    "gcn": {
        "hidden_dim": 64,
        "num_layers": 2,
        "input_dropout": 0.2,
        "dropout": 0.2,
        "nonlinearity": "relu",
        "lr": 5e-4,
        "weight_decay": 1e-4,
    },
    "mixhop": {
        "hidden_dims_csv": "30,30",
        "adj_pows": "0,1",
        "output_layer": "fc",
        "input_dropout": 0.5,
        "layer_dropout": 0.5,
        "nonlinearity": "relu",
        "lr": 5e-4,
        "weight_decay": 1e-4,
    },
    "dir_mixhop": {
        "hidden_dims_csv": "120,120",
        "adj_pows": "0,1,2",
        "output_layer": "fc",
        "input_dropout": 0.2,
        "layer_dropout": 0.2,
        "nonlinearity": "relu",
        "alpha": 0.5,
        "learn_alpha": True,
        "lr": 5e-3,
        "weight_decay": 1e-4,
    },
}

In [ ]:
def quick_tune_current_dataset(max_epochs=1000, patience=200, verbose=False):
    rows = []

    for model_name, config in HYPERPARAMS.items():
        for seed in TUNE_SEEDS:
            out = run_one(
                dataset_name=CURRENT_DATASET,
                model_name=model_name,
                config=config,
                split_index=TUNE_SPLIT_INDEX,
                split_seed=CURRENT_SPLIT_SEED,
                seed=seed,
                max_epochs=max_epochs,
                patience=patience,
                root=CURRENT_ROOT,
                verbose=verbose,
            )
            rows.append(out)

    df = pd.DataFrame(rows)
    display(df[["dataset", "model", "seed", "best_train", "best_val", "best_test", "best_epoch", "params"]])
    return df


tune_df = quick_tune_current_dataset(verbose=True)

In [ ]:
def save_current_hyperparams():
    out_dir = RESULTS_DIR / "configs"
    out_dir.mkdir(parents=True, exist_ok=True)
    path = out_dir / f"{CURRENT_DATASET}__hyperparams.json"
    with open(path, "w") as f:
        json.dump(HYPERPARAMS, f, indent=2)
    print(f"Saved hyperparams to {path}")


save_current_hyperparams()

In [ ]:
def get_splits_for_dataset(dataset_name):
    if dataset_name in {"chameleon_filtered_directed", "squirrel_filtered_directed"} and USE_ALL_SPLITS_FOR_FINAL:
        return list(range(10))
    return [TUNE_SPLIT_INDEX]


def run_all_models_current_dataset(max_epochs=1000, patience=200):
    rows = []
    split_list = get_splits_for_dataset(CURRENT_DATASET)

    for split_index in split_list:
        for seed in FINAL_SEEDS:
            for model_name, config in HYPERPARAMS.items():
                print(f"Running {CURRENT_DATASET} | {model_name} | split={split_index} | seed={seed}")
                out = run_one(
                    dataset_name=CURRENT_DATASET,
                    model_name=model_name,
                    config=config,
                    split_index=split_index,
                    split_seed=CURRENT_SPLIT_SEED,
                    seed=seed,
                    max_epochs=max_epochs,
                    patience=patience,
                    root=CURRENT_ROOT,
                    verbose=False,
                )
                rows.append(out)

    df = pd.DataFrame(rows)

    out_dir = RESULTS_DIR / "final"
    out_dir.mkdir(parents=True, exist_ok=True)

    runs_path = out_dir / f"{CURRENT_DATASET}__runs.csv"
    df.to_csv(runs_path, index=False)

    summary = (
        df.groupby("model", as_index=False)
          .agg(
              mean_train=("best_train", "mean"),
              std_train=("best_train", "std"),
              mean_val=("best_val", "mean"),
              std_val=("best_val", "std"),
              mean_test=("best_test", "mean"),
              std_test=("best_test", "std"),
              mean_epoch=("best_epoch", "mean"),
              runs=("best_test", "count"),
          )
          .sort_values("mean_val", ascending=False)
    )

    summary_path = out_dir / f"{CURRENT_DATASET}__summary.csv"
    summary.to_csv(summary_path, index=False)

    print(f"Saved runs to {runs_path}")
    print(f"Saved summary to {summary_path}")
    display(summary)

    return df, summary


final_df, final_summary = run_all_models_current_dataset()

## Alpha ablation

In [ ]:
import copy
import pandas as pd
from pathlib import Path

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

def run_alpha_ablation(
    alphas=(0.0, 0.25, 0.38, 0.5, 0.75, 1.0),
    seeds=[0],
    max_epochs=400,
    patience=80,
):
    rows = []

    base_config = copy.deepcopy(HYPERPARAMS["dir_mixhop"])

    for alpha in alphas:
        config = copy.deepcopy(base_config)
        config["alpha"] = float(alpha)

        for seed in seeds:
            print(f"[ALPHA] {CURRENT_DATASET} | alpha={alpha} | seed={seed}")
            out = run_one(
                dataset_name=CURRENT_DATASET,
                model_name="dir_mixhop",
                config=config,
                split_index=TUNE_SPLIT_INDEX,
                split_seed=CURRENT_SPLIT_SEED,
                seed=seed,
                max_epochs=max_epochs,
                patience=patience,
                root=CURRENT_ROOT,
                verbose=False,
            )
            out["alpha"] = alpha
            rows.append(out)

    df = pd.DataFrame(rows)

    out_dir = RESULTS_DIR / "alpha_ablation"
    out_dir.mkdir(parents=True, exist_ok=True)

    runs_path = out_dir / f"{CURRENT_DATASET}__alpha_runs.csv"
    df.to_csv(runs_path, index=False)

    summary = (
        df.groupby("alpha", as_index=False)
          .agg(
              mean_train=("best_train", "mean"),
              std_train=("best_train", "std"),
              mean_val=("best_val", "mean"),
              std_val=("best_val", "std"),
              mean_test=("best_test", "mean"),
              std_test=("best_test", "std"),
              mean_epoch=("best_epoch", "mean"),
              runs=("best_test", "count"),
          )
          .sort_values("alpha")
    )

    summary_path = out_dir / f"{CURRENT_DATASET}__alpha_quick_summary.csv"
    summary.to_csv(summary_path, index=False)

    print(f"Saved runs to {runs_path}")
    print(f"Saved summary to {summary_path}")
    display(summary)

    return df, summary

alpha_quick_df, alpha_quick_summary = run_alpha_ablation()

In [ ]:
def run_learned_alpha_once(
    max_epochs=400,
    patience=80,
):
    rows = []

    base_config = copy.deepcopy(HYPERPARAMS["dir_mixhop"])
    base_config["learn_alpha"] = True

    for seed in [0]:
        print(f"[LEARNED ALPHA] {CURRENT_DATASET} | seed={seed}")

        out = run_one(
            dataset_name=CURRENT_DATASET,
            model_name="dir_mixhop",
            config=base_config,
            split_index=TUNE_SPLIT_INDEX,
            split_seed=CURRENT_SPLIT_SEED,
            seed=seed,
            max_epochs=max_epochs,
            patience=patience,
            root=CURRENT_ROOT,
            verbose=False,
        )

        learned_alpha = out.get("alpha", None)

        out["alpha"] = learned_alpha
        out["is_learned"] = True

        rows.append(out)

    df = pd.DataFrame(rows)

    summary = (
        df.groupby("alpha", as_index=False)
          .agg(
              mean_test=("best_test", "mean"),
              std_test=("best_test", "std"),
          )
    )

    return df, summary

learned_alpha_df, learned_alpha_summary = run_learned_alpha_once()

In [ ]:
def merge_alpha_results(fixed_summary, learned_summary):
    learned_summary = learned_summary.copy()
    learned_summary["is_learned"] = True

    fixed_summary = fixed_summary.copy()
    fixed_summary["is_learned"] = False

    combined = pd.concat([fixed_summary, learned_summary], ignore_index=True)
    combined = combined.sort_values("alpha")

    return combined

combined_alpha_summary = merge_alpha_results(alpha_quick_summary, learned_alpha_summary)
combined_alpha_summary

In [ ]:
import matplotlib.pyplot as plt

def plot_alpha(df, dataset_name=None):
    fixed = df[df["is_learned"] == False]
    learned = df[df["is_learned"] == True]

    plt.figure(figsize=(5.5, 3.8))

    plt.plot(fixed["alpha"], fixed["mean_test"], marker="o", linewidth=2)

    if len(learned) > 0:
        plt.scatter(
            learned["alpha"],
            learned["mean_test"],
            s=100,
            marker="*",
            label="Learned α",
            zorder=5,
        )

    plt.xlabel(r"$\alpha$")
    plt.ylabel("Test Accuracy")

    if dataset_name:
        plt.title(f"Alpha Ablation ({dataset_name})")

    plt.grid(True, linestyle="--", linewidth=0.5, alpha=0.4)

    if len(learned) > 0:
        plt.legend()

    plt.tight_layout()
    plt.savefig("alpha_fig.png")
    plt.show()


plot_alpha(combined_alpha_summary, dataset_name=CURRENT_DATASET)